# Open, encode, decode, and visualize a 3D sequence

This uses the real `4d_files/Rafa_Approves_hd_4k` OBJ sequence, five modular in-process codecs, exact round-trip checks, and the public Open4D visualizer.

In [ ]:
import os
import time
from pathlib import Path
import numpy as np

from open4d import MemoryFrameProvider, Sequence
from open4d.codec import decode_sequence, encode_sequence
from open4d.io import inspect_sequence, open_sequence
from open4d.visualization import visualize

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "4d_files/Rafa_Approves_hd_4k").is_dir()
)
DATASET = ROOT / "4d_files/Rafa_Approves_hd_4k"
ARTIFACT_DIR = Path(os.environ.get("OPEN4D_ARTIFACT_DIR", ROOT / ".context/rafa_codecs"))
DEMO_FRAMES = int(os.environ.get("OPEN4D_DEMO_FRAMES", "10")) or None
CODECS = ("raw", "deflate", "bzip2", "lzma", "rle")

In [ ]:
info = inspect_sequence(DATASET)
sequence = open_sequence(DATASET, fps=30)
selected = sequence if DEMO_FRAMES is None else sequence[:DEMO_FRAMES]
demo = Sequence(MemoryFrameProvider(
    tuple(selected), metadata=selected.metadata, topology=selected.topology,
    has_constant_vertex_count=selected.has_constant_vertex_count,
    has_vertex_correspondence=selected.has_vertex_correspondence,
))
print(f"Loaded {info.frame_count} {info.format.upper()} frames; using {len(demo)} for this run.")

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
results = []
decoded = None
for codec in CODECS:
    artifact = ARTIFACT_DIR / f"rafa-{codec}.o4d"
    start = time.perf_counter()
    encode_sequence(demo, artifact, codec=codec, overwrite=True)
    encode_s = time.perf_counter() - start
    candidate = decode_sequence(artifact)  # inferred from the artifact manifest
    assert candidate.metadata == demo.metadata
    assert candidate.topology is demo.topology
    start = time.perf_counter()
    for expected, actual in zip(demo, candidate, strict=True):
        assert (actual.frame_index, actual.timestamp, actual.metadata) == (
            expected.frame_index, expected.timestamp, expected.metadata
        )
        np.testing.assert_array_equal(actual.geometry.positions, expected.geometry.positions)
        np.testing.assert_array_equal(actual.geometry.triangles, expected.geometry.triangles)
    decode_s = time.perf_counter() - start
    results.append((codec, artifact.stat().st_size, encode_s, decode_s))
    if codec == "deflate":
        decoded = candidate
    else:
        candidate.close()

In [ ]:
print("codec      size (MB)  encode (s)  decode+verify (s)")
for codec, size, encode_s, decode_s in results:
    print(f"{codec:<10} {size / 1_000_000:>9.2f}  {encode_s:>10.3f}  {decode_s:>17.3f}")
print(f"Verified {len(decoded)} decoded frames exactly; visualizing DEFLATE.")

In [ ]:
if os.environ.get("OPEN4D_NOTEBOOK_HEADLESS") == "1":
    print("Headless run: visualization call skipped.")
else:
    visualize(decoded, up="y", fps=30)
decoded.close()